# Merge, Join, Concatenate e Compare — pandas 3.0.3

O pandas oferece diversos métodos para combinar e comparar objetos `Series` ou `DataFrame`:

| Função / Método | Descrição |
|---|---|
| `concat()` | Combina múltiplos `Series` ou `DataFrame` ao longo de um índice ou coluna compartilhado |
| `DataFrame.join()` | Combina múltiplos `DataFrame` pelas colunas |
| `DataFrame.combine_first()` | Preenche valores ausentes com os valores não-ausentes de outro `DataFrame` |
| `merge()` | Combina dois `Series` ou `DataFrame` com join no estilo SQL |
| `merge_ordered()` | Combina dois `Series` ou `DataFrame` ao longo de um eixo ordenado |
| `merge_asof()` | Combina dois objetos por correspondência aproximada em vez de exata |
| `Series.compare()` / `DataFrame.compare()` | Mostra diferenças de valores entre dois objetos |


In [ ]:
import numpy as np
import pandas as pd

---
## `concat()`

A função `concat()` concatena uma quantidade arbitrária de objetos `Series` ou `DataFrame` ao longo de um eixo, realizando opcionalmente lógica de conjuntos (união ou interseção) nos demais eixos. Assim como `numpy.concatenate`, `concat()` recebe uma lista ou dicionário de objetos homogêneos e os concatena.

### Exemplo básico de concatenação por linhas

In [ ]:
# Criando três DataFrames com índices sequenciais diferentes
df1 = pd.DataFrame(
    {
        "A": ["A0", "A1", "A2", "A3"],
        "B": ["B0", "B1", "B2", "B3"],
        "C": ["C0", "C1", "C2", "C3"],
        "D": ["D0", "D1", "D2", "D3"],
    },
    index=[0, 1, 2, 3],
)

df2 = pd.DataFrame(
    {
        "A": ["A4", "A5", "A6", "A7"],
        "B": ["B4", "B5", "B6", "B7"],
        "C": ["C4", "C5", "C6", "C7"],
        "D": ["D4", "D5", "D6", "D7"],
    },
    index=[4, 5, 6, 7],
)

df3 = pd.DataFrame(
    {
        "A": ["A8", "A9", "A10", "A11"],
        "B": ["B8", "B9", "B10", "B11"],
        "C": ["C8", "C9", "C10", "C11"],
        "D": ["D8", "D9", "D10", "D11"],
    },
    index=[8, 9, 10, 11],
)

# Agrupa os DataFrames em uma lista e concatena
frames = [df1, df2, df3]
result = pd.concat(frames)
result

> **Nota:** `concat()` faz uma cópia completa dos dados. Reutilizar `concat()` iterativamente pode criar cópias desnecessárias. Colete todos os objetos `DataFrame` ou `Series` em uma lista antes de usar `concat()`:
>
> ```python
> frames = [processar_arquivo(f) for f in arquivos]
> result = pd.concat(frames)
> ```

> **Nota:** Ao concatenar `DataFrames` com eixos nomeados, o pandas tentará preservar os nomes de índice/coluna sempre que possível. Se todas as entradas compartilharem um nome comum, esse nome será atribuído ao resultado. Quando os nomes divergirem, o resultado ficará sem nome.

---
### Lógica de junção do eixo resultante

O parâmetro `join` especifica como tratar valores de eixo que não existem no primeiro `DataFrame`.

- **`join='outer'`** (padrão): toma a **união** de todos os valores de eixo.
- **`join='inner'`**: toma a **interseção** dos valores de eixo.

In [ ]:
# DataFrame com colunas diferentes de df1
df4 = pd.DataFrame(
    {
        "B": ["B2", "B3", "B6", "B7"],
        "D": ["D2", "D3", "D6", "D7"],
        "F": ["F2", "F3", "F6", "F7"],
    },
    index=[2, 3, 6, 7],
)

# join='outer' (padrão): mantém todos os índices, preenche com NaN onde não há dados
result = pd.concat([df1, df4], axis=1)
result

In [ ]:
# join='inner': mantém apenas os índices comuns aos dois DataFrames
result = pd.concat([df1, df4], axis=1, join="inner")
result

Para realizar um "left join" efetivo usando o índice exato do `DataFrame` original, pode-se usar `reindex()`:

In [ ]:
# Reindexação para simular left join: mantém apenas o índice de df1
result = pd.concat([df1, df4], axis=1).reindex(df1.index)
result

### Ignorando índices na concatenação

Para `DataFrames` cujo índice não tem significado, `ignore_index=True` ignora índices sobrepostos e cria um novo índice sequencial:

In [ ]:
# ignore_index=True: reinicia o índice do resultado; sort=False mantém a ordem das colunas
result = pd.concat([df1, df4], ignore_index=True, sort=False)
result

### Concatenando Series e DataFrame juntos

É possível concatenar uma mistura de `Series` e `DataFrame`. A `Series` será transformada em `DataFrame` usando o nome da `Series` como nome da coluna. `Series` sem nome serão numeradas consecutivamente.

In [ ]:
# Series nomeada se torna coluna 'X' no resultado
s1 = pd.Series(["X0", "X1", "X2", "X3"], name="X")
result = pd.concat([df1, s1], axis=1)
result

In [ ]:
# Series sem nome recebem números consecutivos (0, 1, 2...)
s2 = pd.Series(["_0", "_1", "_2", "_3"])
result = pd.concat([df1, s2, s2, s2], axis=1)
result

In [ ]:
# ignore_index=True descarta todas as referências de nome nas colunas
result = pd.concat([df1, s1], axis=1, ignore_index=True)
result

### Chaves resultantes (`keys`)

O argumento `keys` adiciona outro nível de eixo ao índice ou coluna resultante (criando um `MultiIndex`), associando chaves específicas a cada `DataFrame` original.

In [ ]:
# keys cria um MultiIndex: o primeiro nível identifica a origem de cada bloco
result = pd.concat(frames, keys=["x", "y", "z"])
result

In [ ]:
# O MultiIndex permite seleção direta por chave
result.loc["y"]

In [ ]:
# keys pode sobrescrever os nomes das colunas ao criar DataFrame a partir de Series
s3 = pd.Series([0, 1, 2, 3], name="foo")
s4 = pd.Series([0, 1, 2, 3])
s5 = pd.Series([0, 1, 4, 5])

# Sem keys: usa nome da Series ou número sequencial
pd.concat([s3, s4, s5], axis=1)

In [ ]:
# Com keys: define nomes explícitos para as colunas
pd.concat([s3, s4, s5], axis=1, keys=["red", "blue", "yellow"])

### Passando um dicionário para `concat()`

É possível passar um dicionário para `concat()`. Nesse caso, as chaves do dicionário serão usadas como argumento `keys`:

In [ ]:
# Dicionário: as chaves do dict viram as chaves do MultiIndex
pieces = {"x": df1, "y": df2, "z": df3}
result = pd.concat(pieces)
result

In [ ]:
# Passando keys explicitamente com dict: filtra e reordena
result = pd.concat(pieces, keys=["z", "y"])
result

### Níveis do MultiIndex (`levels`)

O `MultiIndex` criado possui níveis construídos a partir das `keys` passadas e do índice dos pedaços do `DataFrame`. O argumento `levels` permite especificar os níveis resultantes associados às `keys`:

In [ ]:
# Inspecionando os níveis do MultiIndex gerado
result.index.levels

In [ ]:
# levels permite definir níveis adicionais no MultiIndex (mesmo que não apareçam nos dados)
result = pd.concat(
    pieces,
    keys=["x", "y", "z"],
    levels=[["z", "y", "x", "w"]],
    names=["group_key"]
)
result

In [ ]:
# O nível 'w' existe no índice mas não foi utilizado nos dados
result.index.levels

### Adicionando linhas a um DataFrame

Se você tem uma `Series` que deseja adicionar como uma única linha a um `DataFrame`, converta a linha em `DataFrame` e use `concat()`:

In [ ]:
# Series com índice compatível com as colunas de df1
s2 = pd.Series(["X0", "X1", "X2", "X3"], index=["A", "B", "C", "D"])

# Converte Series para DataFrame transposto e concatena
# ignore_index=True reinicia a numeração das linhas
result = pd.concat([df1, s2.to_frame().T], ignore_index=True)
result

---
## `merge()`

`merge()` realiza operações de join semelhantes a bancos de dados relacionais como SQL. Usuários familiarizados com SQL mas novos no pandas podem consultar a [comparação com SQL](https://pandas.pydata.org/docs/getting_started/comparison/comparison_with_sql.html).

### Tipos de Merge

`merge()` implementa as operações comuns de join no estilo SQL:

- **one-to-one**: join de dois `DataFrames` pelos índices, que devem conter valores únicos.
- **many-to-one**: join de um índice único com uma ou mais colunas de outro `DataFrame`.
- **many-to-many**: join de colunas com colunas.

> **Nota:** Ao fazer join de colunas com colunas (potencialmente many-to-many), os índices dos `DataFrames` passados serão descartados. Para um join many-to-many, se uma combinação de chave aparecer mais de uma vez em ambas as tabelas, o resultado será o **produto cartesiano** dos dados associados.

### O argumento `how`

O argumento `how` especifica quais chaves são incluídas na tabela resultante. Se uma combinação de chave não aparecer em nenhuma das tabelas, os valores serão `NA`.

| Método de Merge | Equivalente SQL | Descrição |
|---|---|---|
| `left` | LEFT OUTER JOIN | Usa apenas as chaves do DataFrame esquerdo |
| `right` | RIGHT OUTER JOIN | Usa apenas as chaves do DataFrame direito |
| `outer` | FULL OUTER JOIN | Usa a união das chaves de ambos |
| `inner` | INNER JOIN | Usa a interseção das chaves de ambos |
| `cross` | CROSS JOIN | Cria o produto cartesiano das linhas de ambos |

#### Exemplo básico (inner join em uma chave)

In [ ]:
# DataFrames com chave simples 'key'
left = pd.DataFrame(
    {
        "key": ["K0", "K1", "K2", "K3"],
        "A": ["A0", "A1", "A2", "A3"],
        "B": ["B0", "B1", "B2", "B3"],
    }
)

right = pd.DataFrame(
    {
        "key": ["K0", "K1", "K2", "K3"],
        "C": ["C0", "C1", "C2", "C3"],
        "D": ["D0", "D1", "D2", "D3"],
    }
)

# Merge padrão (inner) pela coluna 'key'
result = pd.merge(left, right, on="key")
result

#### Exemplos com múltiplas chaves e diferentes tipos de join

In [ ]:
# DataFrames com duas chaves
left = pd.DataFrame(
    {
        "key1": ["K0", "K0", "K1", "K2"],
        "key2": ["K0", "K1", "K0", "K1"],
        "A": ["A0", "A1", "A2", "A3"],
        "B": ["B0", "B1", "B2", "B3"],
    }
)

right = pd.DataFrame(
    {
        "key1": ["K0", "K1", "K1", "K2"],
        "key2": ["K0", "K0", "K0", "K0"],
        "C": ["C0", "C1", "C2", "C3"],
        "D": ["D0", "D1", "D2", "D3"],
    }
)

# Left join: mantém todas as linhas do DataFrame esquerdo
# Onde não há correspondência no direito → NaN
result = pd.merge(left, right, how="left", on=["key1", "key2"])
result

In [ ]:
# Right join: mantém todas as linhas do DataFrame direito
# Onde não há correspondência no esquerdo → NaN
result = pd.merge(left, right, how="right", on=["key1", "key2"])
result

In [ ]:
# Outer join: mantém todas as linhas de ambos os DataFrames
# Valores ausentes recebem NaN
result = pd.merge(left, right, how="outer", on=["key1", "key2"])
result

In [ ]:
# Inner join: mantém apenas as linhas com chaves em ambos os DataFrames
result = pd.merge(left, right, how="inner", on=["key1", "key2"])
result

In [ ]:
# Cross join: produto cartesiano — combina cada linha do esquerdo com cada linha do direito
result = pd.merge(left, right, how="cross")
result

### Merge de Series com MultiIndex

É possível fazer merge de uma `Series` e um `DataFrame` com `MultiIndex`, desde que os nomes do `MultiIndex` correspondam às colunas do `DataFrame`. Também é possível transformar a `Series` em `DataFrame` usando `Series.reset_index()` antes do merge:

In [ ]:
# DataFrame simples com colunas 'Let' e 'Num'
df = pd.DataFrame({"Let": ["A", "B", "C"], "Num": [1, 2, 3]})
df

In [ ]:
# Series com MultiIndex cujos nomes ('Let', 'Num') correspondem às colunas do df
ser = pd.Series(
    ["a", "b", "c", "d", "e", "f"],
    index=pd.MultiIndex.from_arrays(
        [["A", "B", "C"] * 2, [1, 2, 3, 4, 5, 6]], names=["Let", "Num"]
    ),
)
ser

In [ ]:
# reset_index() converte o MultiIndex em colunas para o merge
pd.merge(df, ser.reset_index(), on=["Let", "Num"])

### Chaves de merge duplicadas

> **Atenção:** Fazer merge com chaves duplicadas aumenta significativamente as dimensões do resultado e pode causar overflow de memória.

In [ ]:
# Outer join com chaves duplicadas: gera produto cartesiano
left = pd.DataFrame({"A": [1, 2], "B": [2, 2]})
right = pd.DataFrame({"A": [4, 5, 6], "B": [2, 2, 2]})

result = pd.merge(left, right, on="B", how="outer")
result

### Validação de unicidade de chaves (`validate`)

O argumento `validate` verifica a unicidade das chaves de merge **antes** da operação, protegendo contra overflows de memória e duplicações inesperadas de chaves.

No exemplo abaixo, o DataFrame direito tem chaves duplicadas, então `validate='one_to_one'` levanta um erro:

In [ ]:
left = pd.DataFrame({"A": [1, 2], "B": [1, 2]})
right = pd.DataFrame({"A": [4, 5, 6], "B": [2, 2, 2]})

# validate='one_to_one': exige que as chaves sejam únicas em ambos os lados
try:
    result = pd.merge(left, right, on="B", how="outer", validate="one_to_one")
except Exception as e:
    print(f"Erro esperado: {type(e).__name__}")
    print(e)

In [ ]:
# validate='one_to_many': permite duplicatas no lado direito, mas não no esquerdo
pd.merge(left, right, on="B", how="outer", validate="one_to_many")

### Indicador de resultado do merge (`indicator`)

`merge()` aceita o argumento `indicator`. Se `True`, uma coluna do tipo `Categorical` chamada `_merge` é adicionada ao resultado com os seguintes valores:

| Origem da observação | Valor em `_merge` |
|---|---|
| Chave apenas no DataFrame esquerdo | `left_only` |
| Chave apenas no DataFrame direito | `right_only` |
| Chave presente em ambos | `both` |

In [ ]:
df1 = pd.DataFrame({"col1": [0, 1], "col_left": ["a", "b"]})
df2 = pd.DataFrame({"col1": [1, 2, 2], "col_right": [2, 2, 2]})

# indicator=True adiciona coluna '_merge' identificando a origem de cada linha
pd.merge(df1, df2, on="col1", how="outer", indicator=True)

In [ ]:
# String passada como indicator define o nome da coluna indicadora
pd.merge(df1, df2, on="col1", how="outer", indicator="indicator_column")

### Colunas de valor sobrepostas (`suffixes`)

O argumento `suffixes` recebe uma tupla ou lista de strings para adicionar como sufixo aos nomes de colunas sobrepostos, evitando ambiguidade:

In [ ]:
left = pd.DataFrame({"k": ["K0", "K1", "K2"], "v": [1, 2, 3]})
right = pd.DataFrame({"k": ["K0", "K0", "K3"], "v": [4, 5, 6]})

# Sem suffixes: pandas usa '_x' e '_y' por padrão
result = pd.merge(left, right, on="k")
result

In [ ]:
# Com suffixes customizados: define sufixos explícitos para colunas com mesmo nome
result = pd.merge(left, right, on="k", suffixes=("_l", "_r"))
result

---
## `DataFrame.join()`

`DataFrame.join()` combina as colunas de múltiplos `DataFrames`, potencialmente com índices diferentes, em um único `DataFrame` resultante.

### Join básico pelos índices

In [ ]:
# DataFrames com índices diferentes
left = pd.DataFrame(
    {"A": ["A0", "A1", "A2"], "B": ["B0", "B1", "B2"]},
    index=["K0", "K1", "K2"]
)

right = pd.DataFrame(
    {"C": ["C0", "C2", "C3"], "D": ["D0", "D2", "D3"]},
    index=["K0", "K2", "K3"]
)

# Left join padrão: mantém todos os índices do DataFrame esquerdo
result = left.join(right)
result

In [ ]:
# Outer join: mantém todos os índices de ambos os DataFrames
result = left.join(right, how="outer")
result

In [ ]:
# Inner join: mantém apenas os índices comuns aos dois DataFrames
result = left.join(right, how="inner")
result

### Join com o argumento `on`

`DataFrame.join()` aceita um argumento opcional `on`, que pode ser o nome de uma coluna ou várias colunas para alinhar o `DataFrame` passado:

In [ ]:
# DataFrame esquerdo com coluna 'key'; DataFrame direito com índice de strings
left = pd.DataFrame(
    {
        "A": ["A0", "A1", "A2", "A3"],
        "B": ["B0", "B1", "B2", "B3"],
        "key": ["K0", "K1", "K0", "K1"],
    }
)

right = pd.DataFrame(
    {"C": ["C0", "C1"], "D": ["D0", "D1"]},
    index=["K0", "K1"]
)

# on='key': usa a coluna 'key' do esquerdo para alinhar com o índice do direito
result = left.join(right, on="key")
result

In [ ]:
# Equivalente usando merge com left_on e right_index
result = pd.merge(
    left, right, left_on="key", right_index=True, how="left", sort=False
)
result

### Join com múltiplas chaves e MultiIndex

Para fazer join em múltiplas chaves, o `DataFrame` passado deve ter um `MultiIndex`:

In [ ]:
# DataFrame esquerdo com duas colunas de chave
left = pd.DataFrame(
    {
        "A": ["A0", "A1", "A2", "A3"],
        "B": ["B0", "B1", "B2", "B3"],
        "key1": ["K0", "K0", "K1", "K2"],
        "key2": ["K0", "K1", "K0", "K1"],
    }
)

# DataFrame direito com MultiIndex
index = pd.MultiIndex.from_tuples(
    [("K0", "K0"), ("K1", "K0"), ("K2", "K0"), ("K2", "K1")]
)
right = pd.DataFrame(
    {"C": ["C0", "C1", "C2", "C3"], "D": ["D0", "D1", "D2", "D3"]},
    index=index
)

# Join usando as colunas 'key1' e 'key2' do esquerdo como chaves
# Tipo padrão: left join
result = left.join(right, on=["key1", "key2"])
result

In [ ]:
# Inner join com múltiplas chaves
result = left.join(right, on=["key1", "key2"], how="inner")
result

### Join de um Index simples com um MultiIndex

É possível fazer join de um `DataFrame` com `Index` simples com um `DataFrame` que possui `MultiIndex`, usando um dos níveis. O nome do `Index` deve coincidir com o nome do nível do `MultiIndex`:

In [ ]:
# DataFrame esquerdo com Index nomeado 'key'
left = pd.DataFrame(
    {"A": ["A0", "A1", "A2"], "B": ["B0", "B1", "B2"]},
    index=pd.Index(["K0", "K1", "K2"], name="key")
)

# DataFrame direito com MultiIndex cujo primeiro nível também se chama 'key'
index = pd.MultiIndex.from_tuples(
    [("K0", "Y0"), ("K1", "Y1"), ("K2", "Y2"), ("K2", "Y3")],
    names=["key", "Y"],
)
right = pd.DataFrame(
    {"C": ["C0", "C1", "C2", "C3"], "D": ["D0", "D1", "D2", "D3"]},
    index=index
)

# O join usa o nível 'key' comum aos dois índices
result = left.join(right, how="inner")
result

### Join com dois MultiIndex

O `MultiIndex` do argumento passado deve ser completamente utilizado no join e deve ser um subconjunto dos índices do argumento à esquerda:

In [ ]:
# DataFrame esquerdo com MultiIndex de 3 níveis (abc, xy, num)
leftindex = pd.MultiIndex.from_product(
    [list("abc"), list("xy"), [1, 2]], names=["abc", "xy", "num"]
)
left = pd.DataFrame({"v1": range(12)}, index=leftindex)
left

In [ ]:
# DataFrame direito com MultiIndex de 2 níveis (abc, xy) — subconjunto do esquerdo
rightindex = pd.MultiIndex.from_product(
    [list("abc"), list("xy")], names=["abc", "xy"]
)
right = pd.DataFrame(
    {"v2": [100 * i for i in range(1, 7)]}, index=rightindex
)
right

In [ ]:
# Join usando os níveis 'abc' e 'xy' comuns aos dois DataFrames
left.join(right, on=["abc", "xy"], how="inner")

### Merge combinando colunas e níveis de índice

Os parâmetros `on`, `left_on` e `right_on` podem referenciar tanto nomes de colunas quanto nomes de níveis de índice. Isso permite fazer merge de instâncias de `DataFrame` em uma combinação de níveis de índice e colunas sem redefinir os índices.

> **Nota:** Quando `DataFrames` são combinados usando apenas alguns níveis de um `MultiIndex`, os níveis extras serão descartados do join resultante. Para preservá-los, use `DataFrame.reset_index()` nesses níveis antes do join.

In [ ]:
# DataFrame esquerdo com MultiIndex de 2 níveis e coluna 'A', 'B'
leftindex = pd.MultiIndex.from_tuples(
    [("K0", "X0"), ("K0", "X1"), ("K1", "X2")], names=["key", "X"]
)
left = pd.DataFrame(
    {"A": ["A0", "A1", "A2"], "B": ["B0", "B1", "B2"]}, index=leftindex
)

# DataFrame direito com MultiIndex de 2 níveis e colunas 'C', 'D'
rightindex = pd.MultiIndex.from_tuples(
    [("K0", "Y0"), ("K1", "Y1"), ("K2", "Y2"), ("K2", "Y3")], names=["key", "Y"]
)
right = pd.DataFrame(
    {"C": ["C0", "C1", "C2", "C3"], "D": ["D0", "D1", "D2", "D3"]}, index=rightindex
)

# Merge combinando colunas e índices: reset_index move os índices para colunas
# e set_index recria o índice hierárquico após o merge
result = pd.merge(
    left.reset_index(), right.reset_index(), on=["key"], how="inner"
).set_index(["key", "X", "Y"])
result

In [ ]:
# Merge usando índice e coluna simultaneamente como chave (sem reset_index)
# Os parâmetros on, left_on e right_on aceitam nomes de níveis de índice
left_index = pd.Index(["K0", "K0", "K1", "K2"], name="key1")
left = pd.DataFrame(
    {
        "A": ["A0", "A1", "A2", "A3"],
        "B": ["B0", "B1", "B2", "B3"],
        "key2": ["K0", "K1", "K0", "K1"],
    },
    index=left_index,
)

right_index = pd.Index(["K0", "K1", "K2", "K2"], name="key1")
right = pd.DataFrame(
    {
        "C": ["C0", "C1", "C2", "C3"],
        "D": ["D0", "D1", "D2", "D3"],
        "key2": ["K0", "K0", "K0", "K1"],
    },
    index=right_index,
)

# Merge referenciando 'key1' (nível de índice) e 'key2' (coluna) ao mesmo tempo
result = left.merge(right, on=["key1", "key2"])
result

### Fazendo join em múltiplos DataFrames

Uma lista ou tupla de `DataFrames` também pode ser passada para `join()` para combiná-los pelos índices:

In [ ]:
# Dois DataFrames adicionais para o join múltiplo
right2 = pd.DataFrame({"v": [7, 8, 9]}, index=["K1", "K1", "K2"])

# Passando uma lista de DataFrames para join em uma única chamada
result = left.join([right, right2])
result

---
## `DataFrame.combine_first()`

`DataFrame.combine_first()` atualiza valores ausentes de um `DataFrame` com os valores **não-ausentes** de outro `DataFrame` na posição correspondente:

In [ ]:
# df1 tem NaN em algumas posições
df1 = pd.DataFrame(
    [[np.nan, 3.0, 5.0], [-4.6, np.nan, np.nan], [np.nan, 7.0, np.nan]]
)

# df2 tem valores alternativos (índice 1 e 2 sobrepostos a df1)
df2 = pd.DataFrame([[-42.6, np.nan, -8.2], [-5.0, 1.6, 4]], index=[1, 2])

# combine_first: preenche NaN do df1 com os valores correspondentes do df2
result = df1.combine_first(df2)
result

---
## `merge_ordered()`

`merge_ordered()` combina dados ordenados — como séries numéricas ou temporais — com preenchimento opcional de dados ausentes via `fill_method`:

In [ ]:
# Dados ordenados com gaps entre as chaves
left = pd.DataFrame(
    {"k": ["K0", "K1", "K1", "K2"], "lv": [1, 2, 3, 4], "s": ["a", "b", "c", "d"]}
)

right = pd.DataFrame(
    {"k": ["K1", "K2", "K4"], "rv": [1, 2, 3]}
)

# merge_ordered: merge com dados ordenados
pd.merge_ordered(left, right, on="k")

In [ ]:
# fill_method='ffill': propagação de valores para frente em dados ausentes
pd.merge_ordered(left, right, on="k", fill_method="ffill")